# CPG-RL 訓練：智元 D1 EDU **輪足版（ZSL-1w）** · MJX · Colab GPU

移植自 task4 的 Go2 論文標準版。四顆輪子是**被動鉸鏈**（受力可轉、有阻尼與靜摩擦，
無致動器），動作空間仍是 12 維。與 Go2 版的差異：

| 項目 | Go2（task4）| D1 輪足（本檔）|
|---|---|---|
| home 關節角 | `[0, 0.9, -1.8]` | `[0, 1.05, -2.00]` |
| PD | `apply_pd()` 覆寫 90/3 | XML 內建 kp=80/kd=1（原廠 demo 值），**無 apply_pd** |
| 力矩上限 | 23.7 / knee 45.43 | 28 全關節 |
| 負重 DR | 0~8 kg | 0~5 kg（官方額定 payload）|
| 總質量 | 15 kg | **20.56 kg**（四顆輪各 0.9 kg）|
| 步幅尺度 | 前後/側向同為 0.12 | 前後 0.12、**側向 0.09**（abad 行程僅 ±28°）|
| obs | 76 維 | **69 維** = 76 −3（機身線速度，實機 LowLevel 拿不到）−4（腳觸地布林）|
| obs 的觸地布林 | 腳掌世界高度 < 3 cm | **移除**（實機無可用訊號，見關卡 3）|
| 高度獎勵基準 | keyframe z | **0.2695**（實際站定高度，非 keyframe 的 0.2948）|
| `done` 低姿門檻 | 0.18（站立高 0.30）| **0.16**（站立高 0.2695，等比例換算）|
| IMU DR | 無 | 重力/角速度加雜訊與偏差 |

執行階段 → 變更類型 → GPU。先跑 Smoke test 再開訓練。

> **本檔所有「Cell N」都是 0-based**（第一格 = Cell 0，也就是這格 markdown）。
> `task6/README.md` 用同一套編號。

---

## reward v4（第四輪訓練）——補上垂直彈跳這個洞

v3 的成果是真的，但只成功了一半：

| | v1 | **v3** |
|---|---|---|
| 週期內俯仰 | 12.00° | **6.75°** ✅ |
| 各腿 omega 變異 | 0.378 | **0.0057** ✅ |
| 相位鎖定 R | 0.993↑ | **1.000（四腿全滿）** ✅ |
| 最弱腿峰值離地 | 38.5 mm | **41.4 mm** ✅ |
| **機身垂直彈跳（峰對峰）** | 49.0 mm | **89.9 mm** ❌ |
| **FR 每週期抬起次數** | 1.99 | **2.03** ❌（正常 1.00）|

問題出在「打地鼠」：v3 罰的是**俯仰（旋轉）**，policy 把能量搬到**垂直平移**去，
而那個自由度當時完全沒人管——`r_h` 用係數 40 時，v3 彈 90 mm 也只讓它掉到 0.9673，
整段彈跳只值 0.016 分。

使用者從側視回報「前腳還是用跳的、兩腳幾乎同時往上」，實測對上了：
**FR 每個步態週期抬起 2.03 次**（FL/RL/RR 都是 1.02~1.09）。
時序比對顯示 FR 那次多餘的小抬（26 mm）發生在**機身衝到最高點 304 mm 的同一刻**——
不是 CPG 叫它抬，是機身彈跳把支撐腳從地上拔起來。**一個根因、兩個症狀。**

（附帶教訓：v3 那輪的監看指標只有 `pitch` 和 `clr`，兩個都在改善，
所以完全沒看出垂直彈跳惡化了 1.8 倍。v4 因此新增 `vz` 監看指標。）

| # | 改動 | 位置 | 值 |
|---|---|---|---|
| 1 | `r_h` 銳化 | Cell 6 | `exp(-40·Δz²)` → **`exp(-400·Δz²)`**（v3 行為下 0.967 → 0.741，扣 0.13）|
| 2 | 新增垂直速度懲罰 | Cell 6 | **`W_VZ = 0.5`** × `qvel[2]²`（v3 vz²=0.2064 → 扣 0.10）|
| 3 | `W_PITCHRATE` 砍半 | Cell 6 | `0.1` → **`0.05`**（v3 速度掉 14%，買回來一些）|
| 4 | 新增 `vz` 監看指標 | Cell 6/8/9 | 目標壓回 0.25 m/s 以下（v1 0.264、v3 0.414）|

`W_PITCH = 20.0`、`W_OMEGA_VAR = 0.5`、`OMEGA_MAX = 2.5` **全部不動**——這三項在 v3 已驗證有效。

**這是一個賭注不是保證**：預期彈跳壓下去後 FR 的雙抬會自然消失（因為支撐腳不再被拔起來）。
若彈跳壓下去了 FR 還是 2 次／週期，那就是另一條線索，要重查。

---

## reward v3（第三輪訓練，保留紀錄）——四項改動

第二輪（v2）的結論是**改錯方向**：`W_SCUFF` 罰對了現象卻可以被規避，而且與目標負相關。
實測 v1→v2 的 scuff 從 0.0976 降到 0.0374（−62%），FR 抬起量 6.7 → 6.6 mm **完全沒動**；
跨三個策略看更清楚——開迴路 scuff 0.288 / 前腳抬 13.2 mm，v2 scuff 0.115 / 前腳只剩 5.4 mm，
**前腳抬得越差、scuff 反而越低**。

2026-08-11 量到真正的根因：**機身與步態同相俯仰，把離地量吃掉**。
前腳在機身座標系擺了 107–112 mm（指令 `G_C` 才 80），只有 38–43 mm 變成真正的離地；
後腳擺 74 mm 卻離地 100 mm。差別在前腳擺到最高點時機身正俯衝 −7~−10°。
懸浮測試（關重力＋關碰撞）顯示純擺腿反作用只佔 3.32°，其餘約 70% 來自地面互動。
而舊 reward **沒有任何一項在管姿態**——機身搖 ±11° 完全免費，
所以 policy 只能在「前腳吃虧」與「後腳吃虧」之間二選一
（開迴路 +3.9° 抬頭 → 後腳慘；v1 −3.9° 低頭 → 前腳慘），無法兩端都好。

第二個必須同時堵的規避管道：policy 把各腿 CPG 頻率拆開
（v1 實測 FL 1.67 / FR 1.75 / RL 1.29 / RR 1.30，前腳比後腳快 30%）。
`W_COUP=8` 的耦合會把相位硬鎖回去，結果不是步態散掉而是**穩定的相位畸變**。
只加俯仰懲罰而不封這個口，下一輪會用同一招。

| # | 改動 | 位置 | 值 |
|---|---|---|---|
| 1 | **移除** `W_SCUFF` 與整個 scuff 項 | Cell 6 | — |
| 2 | 新增俯仰懲罰（姿態 + 角速度）| Cell 6 | `W_PITCH = 20.0`、`W_PITCHRATE = 0.1` |
| 3 | 新增各腿頻率離散懲罰 | Cell 6 | `W_OMEGA_VAR = 0.5` |
| 4 | `OMEGA_MAX` 砍掉倒退走帶 | Cell 4 | `4.5` → **`2.5`** |

權重是照 v1 現況的實測量值訂的，讓每一項在「現在」約值 0.17–0.19（正項總和上限 3.0）：
v1 量到 `grav[0]^2 = 0.0088`、`qvel[4]^2 = 1.7424`、`var(omega) = 0.3785`
（v2 更差：0.0254 / 3.5479 / 0.6313，會自動被罰更重，方向正確）。

改動 4 的依據：kp=80 時 ω≈2.4–3.4 是**倒退走帶**——抬不夠的腿在擺動相被往前拖、
把機身往後推（開迴路 ω=2.4 → −1.33 m）。舊上限 4.5 讓動作空間有一大半落在裡面，
v1 瞬時 ω 最大已用到 3.68。訓好的 policy 常用值只有 1.3–1.75，砍到 2.5 仍有充裕餘裕。

監看指標也跟著換：`scuff` → **`pitch`**（機身俯仰約略角度，越小越好，v1 基準約 4–5°）
與 **`clr`**（四腿平均離地淨空 mm，越大越好，v1 基準約 15 mm、開迴路約 8 mm）。

---

## reward v2（第二輪訓練，已被 v3 取代，保留紀錄）——三項改動

第一輪 120M 訓練「能走但學歪了」：CPG 相位鎖得很好（R≈0.99、trot），
20 秒前進 9.79 m 不跌倒，**但兩隻前腳幾乎不跨步**——
平均抬起量 FL 6.5 mm / FR 6.7 mm（後腳 25.4 / 20.5 mm），
兩隻前腳觸地同步率 0.75（一起動，像在跳），policy 學到的前腳 `fx` 只有 +0.03 / +0.10。

根因：腳的實際位移是 `dx = -D_STEP * fx * cos θ`。
**CPG 的相位鎖只鎖「什麼時候動」，鎖不住「動多少」**——`fx → 0` 時前後步幅被抵消，
前腳只剩上下起伏。針對這個 local optimum 改三項：

| # | 改動 | 位置 | 值 |
|---|---|---|---|
| 1 | 新增擺動相觸地懲罰 `- W_SCUFF * scuff` | Cell 6 | `W_SCUFF = 0.4`（沿用 task4 地形版）|
| 2 | 力矩懲罰係數調降（懷疑是壓抑前腳的元凶）| Cell 6 | `2e-4` → **`5e-5`**（1/4）|
| 3 | 訓練步數減半（第一輪 ~50M 後即為雜訊）| Cell 9 | `120M` → **`60M`**，`num_evals` 維持 20 |

### reward 可以用上帝視角，obs 不行

v3 的 `clr` 監看指標用 `data.geom_xpos` 取**輪心世界高度**，
這是模擬真值、實機拿不到。**這沒有關係**，因為它只在訓練時計算、訓練結束就丟棄：
產物只有 policy 網路權重，而 policy 是純函式 `obs → action`，
本機推論（`task6/inference/local_infer_d1.py`）不會執行到 reward 的任何一行。

反過來 **obs 是推論時每一步都要餵的東西**，只能放實機 LowLevel 真的拿得到的量。
所以輪子淨空高度**絕對不可以**加進 `_obs`——obs 維持 69 維、欄位順序不變。
（v3 的三個 reward 懲罰項用的 `grav[0]` / `qvel[4]` / `om` 則都是實機或本端拿得到的量，
但它們一樣只進 reward 不進 obs，obs 契約完全沒動。）

---

## ⚠️ 常數是刻意的重複——改一邊就要改另一邊

Colab 無法 import 本地模組，所以下面 Cell 4 的常數是
`task6/inference/d1_model.py` 的**手抄副本**。兩份常數一旦分岔，
訓練出來的權重在本機推論時會靜默走樣（維度對得上、行為對不上，不會有任何錯誤訊息）。

**任何一邊改了常數，另一邊必須同步改。** 涉及的常數：

`MU_MIN` `MU_MAX` `OMEGA_MIN` `OMEGA_MAX` `A_CONV` `D_STEP` `D_STEP_Y`
`G_C` `G_P` `NOMINAL_HEIGHT` `W_COUP` `N_CPG_SUB` `CTRL_DT` `SIM_DT`
`KP` `KD` `TAU_MAX` `OBS_DIM` `ACT_DIM` `HOME3` `LEGS` `PHASE_OFFSET`
`FALL_GRAV_Z`（本檔寫成字面值 `-0.4`，在 Cell 6 的 `done` 那行）

（本檔的 `KP_NOM` / `KD_NOM` / `HOME3_np` 對應 d1_model 的 `KP` / `KD` / `HOME3`。）

`_obs` 的**欄位順序**同樣是重複：必須與 `task6/inference/obs_d1.py` 的
`OBS_LAYOUT` 逐項一致，否則權重同樣會靜默失效。

### 反過來：**訓練專用常數**在 d1_model.py 沒有對應項，這是正常的

Cell 6 的 `MIN_HEIGHT`、`W_PITCH`、`W_PITCHRATE`、`W_OMEGA_VAR`、以及 DR 相關的
`PUSH_*` / `*_NOISE` / `GYRO_BIAS` 都**只在 reward / `done` / DR 用**，
推論端不算 reward、不做 DR，所以 `d1_model.py` 找不到它們不是分岔。
唯一的例外是 `WHEEL_RADIUS = 0.0710`：它在本檔只用於 `clr` 監看指標，
但數值與 `d1_model.WHEEL_RADIUS` 和 XML 的輪 geom `size` 是同一個物理量，改到要三邊一起改。

---

## ⚠️ 套件版本也是契約——不要放寬安裝格的版本鎖

`make_ppo_networks` 的 **activation 函式與動作分布類型**沒有在本檔明寫，
取的是 **brax 版本的預設值**。這兩者與本機推論端不一致時，參數形狀完全相同，
brax 載權重**不會報錯**，只會讓 policy 行為靜默錯亂（實測 deterministic
動作偏差 0.59）。相對地，隱藏層大小對不上反而是安全的——會直接丟
`ScopeParamShapeError`。

所以 Cell 1（安裝格）鎖死 `brax==0.14.2`、`mujoco==3.10.0`（本機推論端版本），
Cell 2 再斷言一次。

---

## ⚠️ jax 必須 `<0.10`——而這一項在 Colab GPU 上有不確定性

`brax==0.14.2` 的 `ppo.train`（`train.py:756`）呼叫 `jax.device_put_replicated`，
該 API 在 **jax 0.10 已移除**。本機實測：

| jax | 結果 |
|---|---|
| ≤ 0.8 | 正常 |
| 0.9.x | 正常，只有 DeprecationWarning |
| **0.10.x** | `AttributeError: jax.device_put_replicated is deprecated; use jax.device_put instead.`（訓練起手就死）|

而 `brax==0.14.2` 只要求 `jax>=0.4.6`，**不會**把 Colab 預裝的新 jax 拉回去，
所以 Cell 1 明寫 `"jax[cuda12]<0.10"`，Cell 2 有一個會早死的斷言擋著。

### ❗ 這一項無法在本機驗證，請注意

Colab 的 GPU jax 是**預裝**的，指定 `jax[cuda12]<0.10` **未必**能同時保住 CUDA 支援
（可能裝完 `jax.devices()` 只剩 CPU，或 CUDA plugin 版本解析失敗）。
裝完務必看 Cell 2 印出的 `devices:` 有沒有 `cuda`。

**如果裝不起來 / 掉回 CPU，請回報，不要自行改 `brax` 版本。**
換 brax 版本會改變 `make_ppo_networks` 的預設 activation，
那會讓權重與本機推論端**靜默不匹配**（見上一節），比訓練跑不動更糟。

可以試的備案（依序）：
1. 裝完後「執行階段 → 重新啟動工作階段」再跑 Cell 2（pip 換版常需重啟才生效）。
2. 把 `"jax[cuda12]<0.10"` 改成明確版本，例如 `"jax[cuda12]==0.9.2"`。
3. 都不行 → 回報，由本機端改 brax/推論端版本一起換，兩邊同步。


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

# 版本鎖死，不要放寬。brax 的預設 activation/分布由版本決定，而 activation
# 不匹配時 brax 載權重【不會報錯】，只會讓 policy 靜默錯亂（見本檔開頭說明）。
# 這裡的版本 = 本機推論端 (task6/inference/) 的版本。
#
# jax<0.10：brax 0.14.2 的 ppo.train (train.py:756) 用 jax.device_put_replicated，
# 該 API 在 jax 0.10 已被移除（本機 jax 0.10.2 實測直接 AttributeError）。
# 用 jax[cuda12] 這個 extra 是為了讓 jaxlib 與 CUDA plugin 一起降到相容版本，
# 只寫 "jax<0.10" 會留下版本不合的 jaxlib/cuda plugin。
# ⚠️ 這一項無法在本機驗證：裝完務必看下一格印出的 devices 有沒有 cuda。
#    掉回 CPU 或裝不起來 → 回報，【不要】自行改 brax 版本（會動到 activation 預設值）。
!pip install -q "brax==0.14.2" "mujoco==3.10.0" "mujoco-mjx==3.10.0" "jax[cuda12]<0.10" mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

# 版本斷言：不匹配當場停住，不要繞過。訓練跑完才發現行為對不上就白費了。
import brax, mujoco
print("brax", brax.__version__, "| mujoco", mujoco.__version__)
assert brax.__version__ == "0.14.2", (
    f"brax 版本為 {brax.__version__}，本機推論端是 0.14.2。"
    "版本不同會改變 make_ppo_networks 的預設 activation，"
    "而 activation 不匹配時 brax 載權重【不會報錯】，只會讓 policy 行為錯亂。"
    "請回到上一格重跑安裝（Colab 有時需要「執行階段 → 重新啟動工作階段」才會生效）。"
)
assert mujoco.__version__ == "3.10.0", (
    f"mujoco 版本為 {mujoco.__version__}，本機是 3.10.0。"
    "MJX 的接觸/求解器行為隨版本改變，訓練與推論不同版會讓步態對不上。"
)
# jax 0.10 移除了 device_put_replicated，而 brax 0.14.2 的 ppo.train 會用它。
# 在這裡早死，不要拖到 Cell 9 編譯完才炸。
assert hasattr(jax, "device_put_replicated"), (
    f"jax {jax.__version__} 已移除 device_put_replicated，brax 0.14.2 的 ppo.train 會失敗。"
    "需要 jax<0.10。若 Colab 無法在此版本下取得 GPU 支援，請回報——"
    "換 brax 版本會改變 make_ppo_networks 的預設 activation，那會讓權重與本機推論端靜默不匹配。"
)
# GPU 不是硬性錯誤（CPU 也跑得動，只是慢到不切實際），但一定要看見這行警告
if not any(d.platform == "gpu" for d in jax.devices()):
    print("⚠️ 沒抓到 GPU。確認「執行階段 → 變更類型 → GPU」，"
          "以及上一格的 jax[cuda12]<0.10 是否把 CUDA 支援裝掉了（見本檔開頭的備案）。")
print("版本 OK")

In [ ]:
import os, subprocess

REPO = "https://github.com/HGLLLLL/RBTDOG_SIM.git"
BRANCH = "feat/d1-edu-cpg-rl"    # task6/ 只在這個功能分支；併進 main 後改成 "main"
DEST = "rbtdog_sim"              # 明寫目的地：repo 名是大寫 RBTDOG_SIM，預設會 clone 成別的資料夾

if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST],
                   check=True)
SCENE = f"{DEST}/task6/model/d1_edu_w/scene_mjx.xml"
print("model exists:", os.path.exists(SCENE))

# 抓不到就當場停：Cell 5 才炸的話錯誤訊息會指向 MuJoCo，看不出真正原因
assert os.path.exists(SCENE), (
    f"抓不到 {SCENE}。\n"
    f"1) 確認分支 {BRANCH} 的最新 commit 已 push 上 {REPO}"
    f"（--depth 1 只抓得到遠端當下的內容，本機沒 push 的修正 Colab 看不到）；\n"
    f"2) 或改用左側檔案面板，把整個 task6/model/d1_edu_w/（含 meshes/ 的 17 個 STL）"
    f"上傳到 {DEST}/task6/model/d1_edu_w/。"
)

In [ ]:
import jax.numpy as jnp
import numpy as np

MU_MIN, MU_MAX = 1.0, 2.0
# OMEGA_MAX 4.5 → 2.5（reward v3）。實測 kp=80 時 ω≈2.4–3.4 是「倒退走帶」：
# 抬不夠的腿在擺動相被往前拖、把機身往後推，前進量翻負（開迴路 ω=2.4 → −1.33 m）。
# 舊上限 4.5 讓動作空間有一大半落在那個帶裡（v1 瞬時 ω 最大已用到 3.68）。
# 訓好的 policy 常用值只有 1.3–1.75，砍到 2.5 仍有充裕餘裕。
# ★ 必須與 task6/inference/d1_model.py 的 OMEGA_MAX 同值。
OMEGA_MIN, OMEGA_MAX = 0.0, 2.5
A_CONV = 50.0
D_STEP, D_STEP_Y = 0.12, 0.09    # 前後 / 側向；側向較小是因 abad 行程僅 ±28°
G_C, G_P = 0.08, 0.01
NOMINAL_HEIGHT = 0.2695          # 實際站定高度（keyframe 的 0.2948 是純運動學值）
W_COUP = 8.0
N_CPG_SUB = 4
CTRL_DT, SIM_DT = 0.02, 0.004
KP_NOM, KD_NOM = 80.0, 1.0          # 原廠 demo 值
TAU_MAX = 28.0
OBS_DIM, ACT_DIM = 69, 12
HOME3_np = np.array([0.0, 1.05, -2.00])
HOME12 = jnp.array(list(HOME3_np) * 4)
# 輪子有自己的鉸鏈（不熔死），所以 12 個腿關節在 qpos/qvel 裡不連續。
# 必須與 task6/inference/d1_model.py 的 LEG_QPOS_IDX / LEG_QVEL_IDX 逐項相同。
LEG_QPOS_IDX = jnp.array([7, 8, 9, 11, 12, 13, 15, 16, 17, 19, 20, 21])
LEG_QVEL_IDX = jnp.array([6, 7, 8, 10, 11, 12, 14, 15, 16, 18, 19, 20])
LEG_QPOS_IDX_np = [7, 8, 9, 11, 12, 13, 15, 16, 17, 19, 20, 21]
LEGS = ["FL", "FR", "RL", "RR"]

PHASE_OFFSET = jnp.array([0.0, jnp.pi, jnp.pi, 0.0])
PHI = PHASE_OFFSET[None, :] - PHASE_OFFSET[:, None]


def cpg_init():
    return {"rx": jnp.full(4, 1.5), "rx_d": jnp.zeros(4),
            "ry": jnp.full(4, 1.5), "ry_d": jnp.zeros(4),
            "theta": PHASE_OFFSET}


def cpg_step(c, mux, muy, omega, dt):
    rx, rxd, ry, ryd, th = c["rx"], c["rx_d"], c["ry"], c["ry_d"], c["theta"]
    h = dt / N_CPG_SUB
    for _ in range(N_CPG_SUB):
        rxd = rxd + A_CONV * (A_CONV / 4.0 * (mux - rx) - rxd) * h
        rx = rx + rxd * h
        ryd = ryd + A_CONV * (A_CONV / 4.0 * (muy - ry) - ryd) * h
        ry = ry + ryd * h
        rbar = 0.5 * (rx + ry)
        diff = th[None, :] - th[:, None] - PHI
        coup = jnp.sum(rbar[None, :] * jnp.sin(diff), axis=1)
        th = th + (2.0 * jnp.pi * omega + W_COUP * coup) * h
    return {"rx": rx, "rx_d": rxd, "ry": ry, "ry_d": ryd,
            "theta": jnp.mod(th, 2.0 * jnp.pi)}


def action_to_cpg_cmd(action):
    a = jnp.tanh(action).reshape(4, 3)
    mux = (a[:, 0] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    muy = (a[:, 1] + 1) / 2 * (MU_MAX - MU_MIN) + MU_MIN
    om = (a[:, 2] + 1) / 2 * (OMEGA_MAX - OMEGA_MIN) + OMEGA_MIN
    return mux, muy, om

In [ ]:
import mujoco

def leg_ik_consts(xml):
    m = mujoco.MjModel.from_xml_path(xml); d = mujoco.MjData(m)
    f0s, jinvs = [], []
    for k, leg in enumerate(LEGS):
        jb = LEG_QPOS_IDX_np[3 * k:3 * k + 3]   # 輪關節夾在中間，位址不連續
        gid = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, leg)
        hip = mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_BODY, leg + "_abad")
        def foot(q3):
            mujoco.mj_resetDataKeyframe(m, d, 0)
            d.qpos[jb] = q3; mujoco.mj_forward(m, d)
            return (d.geom_xpos[gid] - d.xpos[hip]).copy()
        f0 = foot(HOME3_np); e = 1e-3; J = np.zeros((3, 3))
        for j in range(3):
            dq = np.zeros(3); dq[j] = e
            J[:, j] = (foot(HOME3_np + dq) - foot(HOME3_np - dq)) / (2 * e)
        f0s.append(f0); jinvs.append(np.linalg.inv(J))
    return np.array(f0s, np.float32), np.array(jinvs, np.float32)

F0S_np, JINVS_np = leg_ik_consts(SCENE)
print("f0 每腿(x,y,z 相對髖):\n", np.round(F0S_np, 3))

In [ ]:
import functools
from brax.envs.base import Env, State
from mujoco import mjx

N_FRAMES = int(round(CTRL_DT / SIM_DT))
PUSH_EVERY = 100          # 每 100 控制步(=2s) 注入一次速度擾動
PUSH_VEL = 0.6
GRAV_NOISE = 0.02         # 重力向量雜訊 sigma（IMU 為原始資料、精度一般）
GYRO_NOISE = 0.10         # 角速度雜訊 sigma (rad/s)
GYRO_BIAS = 0.05          # 每 episode 取樣一次的角速度偏差上限 (rad/s)

# done 的低姿門檻（訓練專用，推論端不需要：local_infer 只用 FALL_GRAV_Z 判跌倒）。
# 由 task4 Go2 的 0.18 等比例換算：0.18 / 0.30(Go2 站立高) * 0.2695(本機 NOMINAL_HEIGHT)
# = 0.1617 → 取 0.16。沒有這條護欄的話，「塌腰趴著慢慢挪」是個很好爬的 local optimum，
# 而本機 base 碰撞 geom 繼承 condim=1（無摩擦肚皮），滑起來幾乎沒有阻力。
MIN_HEIGHT = 0.16

# ---- reward v3：俯仰懲罰 + 頻率離散懲罰；**移除 W_SCUFF** -------------------
# 為什麼拿掉 W_SCUFF（v2 的擺動相觸地懲罰）：它罰對了現象但可以被規避，而且與目標負相關。
#   v1→v2 實測：scuff 0.0976 → 0.0374（降 62%），FR 抬起 6.7 → 6.6 mm（完全沒動）。
#   跨三個策略比對更明顯：開迴路 scuff 0.288 前腳抬 13.2mm ／ v2 scuff 0.115 前腳只剩 5.4mm
#   —— 前腳抬得越差、scuff 反而越低。加大權重只會更用力往錯方向走。
#
# 真正的根因（2026-08-11 量出來的）：**機身與步態同相俯仰，把離地量吃掉**。
#   前腳在機身座標系擺了 107–112 mm（指令 G_C 才 80），但只有 38–43 mm 變成真正的離地；
#   後腳擺 74 mm 卻離地 100 mm。差別在前腳擺到最高點時機身正俯衝 −7~−10°。
#   懸浮測試（關重力＋關碰撞）：純擺腿反作用只佔 3.32°，地面互動佔其餘約 70%。
#   而舊 reward **沒有任何一項在管姿態**，機身搖 ±11° 完全免費，
#   所以 policy 只是在「前腳吃虧」和「後腳吃虧」之間二選一（開迴路 +3.9° 抬頭 → 後腳慘；
#   v1 −3.9° 低頭 → 前腳慘），無法兩端都好。
#
# 第二個規避管道：policy 把各腿 CPG 頻率拆開（v1 實測 FL 1.67 / FR 1.75 / RL 1.29 / RR 1.30，
#   前腳比後腳快 30%）。W_COUP=8 的耦合會把相位硬鎖回去，於是結果不是步態散掉，
#   而是**穩定的相位畸變**——這正是 v2 用來把 scuff 壓下去卻不抬腳的手法。
#   只加俯仰懲罰而不封這個口，下一輪會用同一招，所以兩項必須同時上。
#
# 權重是照 v1 現況的實測量值訂的，讓每一項在「現在」約值 0.17–0.19（正項總和上限 3.0）：
#   量到 v1: grav[0]^2 = 0.0088、qvel[4]^2 = 1.7424、var(omega) = 0.3785
#            （v2 更差：0.0254 / 3.5479 / 0.6313 → 自動被罰更重，方向正確）
W_PITCH     = 20.0   # × grav[0]^2      俯仰姿態；v1 → 0.176（v3 實測有效，不動）
W_PITCHRATE = 0.05   # × qvel[4]^2      俯仰角速度；v3 從 0.1 砍半，把速度買回來（見 v4 說明）
W_OMEGA_VAR = 0.5    # × var(各腿 omega) 封住相位畸變；v3 實測把變異壓到 0.0057（v1 是 0.378）
W_VZ        = 0.5    # × qvel[2]^2      ★v4 新增：機身垂直彈跳；v3 → 0.10
W_CLR       = 1.5    # × r_clr          ★v4 新增：正向抬腳獎勵（reward 從 v1 到 v3 都缺這一項）
                     #   實測 r_clr：開迴路 0.0516、v1 0.1271、v3 0.0856，理論上限約 0.32。
                     #   風險檢查發現：抬腳本身就會晃機身，vz²/r_h 兩項都在跟離地量作對
                     #   （開迴路掃 G_C：抬 13.9mm 罰 0.053 → 抬 63.5mm 罰 0.241，單調遞增）。
                     #   沒有正向獎勵的話，v4 最省事的解就是「乾脆別抬腳」——那是 v2 錯誤方向的重演。
                     #   W_CLR=1.5 讓「抬到位」值 +0.25，蓋過彈跳懲罰增加的 ~0.19，淨獎勵為正。

# ---- reward v4：補上垂直彈跳這個洞 -----------------------------------------
# v3 的結果是「打地鼠」：俯仰（旋轉）確實砍半 12.00° → 6.75°，
# 但 policy 把能量搬到**垂直平移**——那個自由度當時完全沒人管：
#   機身垂直彈跳峰對峰   v1 49.0 mm → v3 **89.9 mm**（惡化 1.8 倍）
#   vz² 平均            v1 0.0923  → v3 **0.2064**
#   而 r_h 用係數 40 時，v3 的行為平均只值 0.9673 → 彈 90 mm 幾乎免費。
#
# 後果是使用者從側視看到的「前腳在跳」：
#   FR 每個步態週期抬起 **2.03 次**（正常 1.00；FL/RL/RR 都是 1.02~1.09）。
#   時序比對顯示 FR 那次多餘的小抬（26 mm）發生在機身衝到最高點 304 mm 的同一刻
#   —— 不是 CPG 叫它抬，是機身彈跳把支撐腳從地上拔起來。一個根因、兩個症狀。
#
# v4 只補這個洞，v3 成功的三項（W_PITCH / W_OMEGA_VAR / OMEGA_MAX=2.5）全部不動。
# 權重照 v3 實測校準，兩項合計約扣 0.23 分（正項總和上限 3.0）：
WHEEL_RADIUS = 0.0710  # 輪半徑(m)；與 d1_model.WHEEL_RADIUS / XML geom size 同值

# 【reward 可以用上帝視角，obs 不行】
# 這裡的觸地判定用 data.geom_xpos（模擬真值的輪心世界高度），實機沒有這個訊號。
# 這不是問題，因為 reward 只在訓練時計算、訓練結束就丟棄：
# 訓練產物只有 policy 網路權重，而 policy 是純函式 obs → action，
# 推論時 (task6/inference/local_infer_d1.py) 根本不會呼叫到 reward 的任何一行。
# 反過來 obs 是推論時每一步都要餵的東西，只能放實機 LowLevel 真的拿得到的量，
# 所以輪子淨空高度這個量【絕對不可以】加進 _obs（obs 維持 69 維、順序不變）。


def _qinv(q): return jnp.array([q[0], -q[1], -q[2], -q[3]])
def _qrot(q, v):
    u = q[1:4]; t = 2.0 * jnp.cross(u, v); return v + q[0] * t + jnp.cross(u, t)
def w2b(quat, v): return _qrot(_qinv(quat), v)


class D1wCpgEnv(Env):
    def __init__(self, f0s, jinvs):
        m = mujoco.MjModel.from_xml_path(SCENE)
        m.opt.timestep = SIM_DT
        self._mj = m
        self.sys = mjx.put_model(m)
        self._init_q = jnp.array(m.key_qpos[0])
        self._lo = jnp.array(m.actuator_ctrlrange[:, 0])
        self._hi = jnp.array(m.actuator_ctrlrange[:, 1])
        self._f0s = jnp.array(f0s)
        self._jinvs = jnp.array(jinvs)
        # 四顆輪 geom 的 id（geom 名就是 FL/FR/RL/RR）。在這裡查一次存起來：
        # mj_name2id 是 Python 端呼叫，放進 step 裡會進不了 jit。
        self._wheel_gids = jnp.array(
            [mujoco.mj_name2id(m, mujoco.mjtObj.mjOBJ_GEOM, leg) for leg in LEGS])

    # ---- 動作 → 關節目標角 ----
    def _joint_targets(self, c):
        th = c["theta"]
        fx = 2 * (c["rx"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
        fy = 2 * (c["ry"] - MU_MIN) / (MU_MAX - MU_MIN) - 1
        dx = -D_STEP * fx * jnp.cos(th)
        dy = D_STEP_Y * fy * jnp.cos(th)
        dz = jnp.where(jnp.sin(th) > 0, G_C * jnp.sin(th), G_P * jnp.sin(th))
        off = jnp.stack([dx, dy, dz], -1)
        q = jax.vmap(lambda ji, o: jnp.array(HOME3_np) + ji @ o)(self._jinvs, off)
        return q.reshape(12)

    # ---- 輪子離地淨空（v4 起同時給 r_clr 與監看指標用；不進 obs，理由見上面說明）----
    def _wheel_clearance(self, data):
        wheel_z = data.geom_xpos[self._wheel_gids, 2]      # 輪心世界高度
        return wheel_z - WHEEL_RADIUS                      # 四腿各自的淨空 (m)

    # ---- 69 維 obs（欄位順序必須與 task6/inference/obs_d1.py 一致）----
    def _obs(self, data, c, cmd, last_a, gyro_bias):
        quat = data.qpos[3:7]
        grav = w2b(quat, jnp.array([0.0, 0.0, -1.0]))
        gyro = data.qvel[3:6] + gyro_bias
        return jnp.concatenate([
            grav, gyro,
            data.qpos[LEG_QPOS_IDX] - HOME12, data.qvel[LEG_QVEL_IDX],
            cmd, last_a,
            c["rx"], c["rx_d"], c["ry"], c["ry_d"],
            jnp.sin(c["theta"]), jnp.cos(c["theta"]),
        ])

    def reset(self, rng):
        k_cmd, k_bias, k_noise = jax.random.split(rng, 3)
        data = mjx.make_data(self.sys).replace(qpos=self._init_q,
                                               ctrl=jnp.array(self._mj.key_ctrl[0]))
        data = mjx.forward(self.sys, data)
        cmd = jnp.array([jax.random.uniform(k_cmd, minval=0.3, maxval=1.0), 0.0, 0.0])
        gyro_bias = jax.random.uniform(k_bias, (3,), minval=-GYRO_BIAS, maxval=GYRO_BIAS)
        c = cpg_init()
        info = {"rng": k_noise, "c": c, "last_a": jnp.zeros(ACT_DIM),
                "cmd": cmd, "gyro_bias": gyro_bias, "step": 0}
        obs = self._obs(data, c, cmd, jnp.zeros(ACT_DIM), gyro_bias)
        # "reward" 這個鍵【必須】有，reset 與 step 都要有，而且兩邊型別要一致。
        # brax 的 EvalWrapper.reset 會就地塞 metrics['reward']（training.py:182），
        # 而 num_evals>1 時訓練開始前就會先跑一次 eval，於是 EpisodeWrapper.step 的
        # jax.lax.scan(carry=state) 會拿到「進去 3 鍵、出來 2 鍵」→ 編譯完當場 TypeError
        # （本機實測：scan body function carry input and carry output must have the
        #  same pytree structure ... symmetric difference of key sets: {'reward'}）。
        # 這是移植 task4 時掉的東西——Go2 版的 metrics 本來就含 "reward"。
        # 同理：step 裡新增任何 metrics 鍵（例如下面的 "pitch"／"clr"／"vz"），這裡也要一起加。
        metrics = {"height": data.qpos[2], "vx": jnp.zeros(()),
                   "reward": jnp.zeros(()), "pitch": jnp.zeros(()),
                   "clr": jnp.zeros(()), "vz": jnp.zeros(())}
        return State(data, obs, jnp.zeros(()), jnp.zeros(()), metrics, info)

    def step(self, state, action):
        info = dict(state.info)
        mux, muy, om = action_to_cpg_cmd(action)
        c = cpg_step(info["c"], mux, muy, om, CTRL_DT)
        q_des = jnp.clip(self._joint_targets(c), self._lo, self._hi)

        data = state.pipeline_state
        rng, k_push, k_dir, k_obs = jax.random.split(info["rng"], 4)
        do_push = (info["step"] % PUSH_EVERY) == (PUSH_EVERY - 1)
        ang = jax.random.uniform(k_dir, minval=0.0, maxval=2 * jnp.pi)
        mag = jax.random.uniform(k_push, minval=0.0, maxval=PUSH_VEL)
        kick = jnp.where(do_push,
                         jnp.array([mag * jnp.cos(ang), mag * jnp.sin(ang), 0.0]),
                         jnp.zeros(3))
        data = data.replace(qvel=data.qvel.at[0:3].add(kick))

        def one(d, _):
            return mjx.step(self.sys, d.replace(ctrl=q_des)), None
        data, _ = jax.lax.scan(one, data, None, length=N_FRAMES)

        grav = w2b(data.qpos[3:7], jnp.array([0.0, 0.0, -1.0]))
        vb = w2b(data.qpos[3:7], data.qvel[0:3])      # reward 用真值速度（只在模擬）
        cmd = info["cmd"]

        r_vx = jnp.exp(-4.0 * (vb[0] - cmd[0]) ** 2)
        r_vy = jnp.exp(-4.0 * vb[1] ** 2)
        r_wz = jnp.exp(-4.0 * (data.qvel[5] - cmd[2]) ** 2)
        # 係數 40 → 400（v4）。40 太鬆：v3 彈 90 mm 也只讓 r_h 掉到 0.9673，
        # 等於整段彈跳只值 0.016 分，policy 當然照彈。改 400 後同樣行為值 0.7411（扣 0.13 分）。
        r_h = jnp.exp(-400.0 * (data.qpos[2] - NOMINAL_HEIGHT) ** 2)  # 用實際站定高度，非 keyframe
        c_act = jnp.sum((action - info["last_a"]) ** 2)
        c_tau = jnp.sum(data.actuator_force ** 2)

        # reward v3 的三個新懲罰項（取代 v2 的 scuff，理由見最上面的常數區註解）。
        # grav 是重力在機身座標系的方向：grav[0] 就是俯仰的正弦，抬頭為負、低頭為正。
        # qvel[4] 是俯仰角速度——freejoint 的 qvel[3:6] 存的是**機身系**角速度（同 gyro）。
        # var(om) 罰「各腿頻率被拆開」，四腿同頻時為 0，不影響正常 trot。
        c_pitch = grav[0] ** 2
        c_pitchrate = data.qvel[4] ** 2
        c_omvar = jnp.var(om)
        # ★v4：機身垂直速度。只銳化 r_h 是罰「位置偏離」，罰不到「來回彈」的動作本身；
        # qvel[2] 直接對付彈跳。freejoint 的 qvel[0:3] 是世界系線速度，z 就是垂直。
        c_vz = data.qvel[2] ** 2

        # ★v4：正向抬腳獎勵。只獎勵「該擺動的腿真的離地了」，
        # 且用 clip(·, 0, 1) 封頂——抬到指令高度 G_C 就給滿分，再高不加分。
        # 封頂是關鍵：v2 當初就是因為懲罰沒有上限，policy 用「擺更大」去規避，
        # 結果機身搖更兇、淨離地反而更差。這裡抬過頭沒有額外好處，
        # 唯一還能加分的方向就是把**最弱的那條腿**拉上來，正好是我們要的。
        swing = jnp.maximum(jnp.sin(c["theta"]), 0.0)
        r_clr = jnp.mean(swing * jnp.clip(self._wheel_clearance(data) / G_C, 0.0, 1.0))

        # tau 係數 2e-4 → 5e-5（降為 1/4，reward v2）。
        # 理由：第一輪前腳 fx 被壓到近 0（+0.03 / +0.10），懷疑力矩懲罰是主因——
        # 跨步要出水平力、水平力要力矩，而「不跨步只上下晃」剛好是最省力矩的解，
        # 於是 c_tau 這一項替 policy 付錢買了那個 local optimum。原值 2e-4 保留在此註解，
        # 若 v2 出現力矩爆衝/抖動再往回調。
        reward = 1.5 * r_vx + 0.5 * r_vy + 0.5 * r_wz + 0.5 * r_h \
                 + W_CLR * r_clr \
                 - 0.01 * c_act - 5e-5 * c_tau \
                 - W_PITCH * c_pitch - W_PITCHRATE * c_pitchrate \
                 - W_OMEGA_VAR * c_omvar - W_VZ * c_vz

        # 兩個終止條件：
        #  (1) 翻倒。-0.4 必須與 d1_model.FALL_GRAV_Z 一致（Colab 無法 import 本地模組，
        #      只能手抄）。兩邊分岔的話，訓練認定「還沒倒」的姿態在本機推論會被
        #      記成跌倒，反之亦然，而且不會有任何錯誤訊息。
        #  (2) 機身太低（塌腰/趴走）。見上面 MIN_HEIGHT 的換算說明。
        fell = grav[2] > -0.4
        too_low = data.qpos[2] < MIN_HEIGHT
        done = jnp.where(jnp.logical_or(fell, too_low), 1.0, 0.0)

        noise = jax.random.normal(k_obs, (6,))
        obs = self._obs(data, c, cmd, action, info["gyro_bias"])
        obs = obs.at[0:3].add(GRAV_NOISE * noise[0:3])
        obs = obs.at[3:6].add(GYRO_NOISE * noise[3:6])

        info.update({"rng": rng, "c": c, "last_a": action, "step": info["step"] + 1})
        # "reward" 鍵不可省，"pitch"／"clr"／"vz" 也要與 reset 對齊，理由見 reset。
        # 三個都是**監看指標不是 reward 項**：pitch 約略角度(度)、clr 四腿平均淨空(mm)、
        # vz 機身垂直速度大小(m/s)。★vz 是 v4 新增——v3 那輪就是因為沒有這個指標，
        # 才會在 pitch 一路下降的同時完全沒看出垂直彈跳惡化了 1.8 倍。
        metrics = {"height": data.qpos[2], "vx": vb[0], "reward": reward,
                   "pitch": jnp.abs(grav[0]) * 57.29578,
                   "clr": jnp.mean(self._wheel_clearance(data)) * 1000.0,
                   "vz": jnp.abs(data.qvel[2])}
        return state.replace(pipeline_state=data, obs=obs, reward=reward,
                             done=done, metrics=metrics, info=info)

    @property
    def observation_size(self): return OBS_DIM

    @property
    def action_size(self): return ACT_DIM

    @property
    def backend(self): return "mjx"

In [ ]:
_mm = mujoco.MjModel.from_xml_path(SCENE)
BASE_ID = mujoco.mj_name2id(_mm, mujoco.mjtObj.mjOBJ_BODY, "base")

def domain_randomize(sys, rng):
    @jax.vmap
    def per_env(rng):
        k1, k2, k3, k4, k5 = jax.random.split(rng, 5)
        geom_friction = sys.geom_friction.at[:, 0].set(
            jax.random.uniform(k1, minval=0.3, maxval=1.0))
        kp = jax.random.uniform(k2, minval=60.0, maxval=100.0)    # 名目 80
        kd = jax.random.uniform(k3, minval=0.5, maxval=2.0)       # 名目 1
        gain = sys.actuator_gainprm.at[:, 0].set(kp)
        bias = sys.actuator_biasprm.at[:, 1].set(-kp).at[:, 2].set(-kd)
        body_mass = sys.body_mass * jax.random.uniform(
            k4, (sys.nbody,), minval=0.8, maxval=1.2)             # 連桿質量 ±20%
        payload = jax.random.uniform(k5, minval=0.0, maxval=5.0)  # 官方額定 payload 5 kg
        body_mass = body_mass.at[BASE_ID].add(payload)
        return geom_friction, gain, bias, body_mass
    gf, gain, bias, bm = per_env(rng)
    in_axes = jax.tree_util.tree_map(lambda x: None, sys)
    in_axes = in_axes.replace(geom_friction=0, actuator_gainprm=0,
                              actuator_biasprm=0, body_mass=0)
    sys = sys.replace(geom_friction=gf, actuator_gainprm=gain,
                      actuator_biasprm=bias, body_mass=bm)
    return sys, in_axes
print("domain_randomize ready")

In [ ]:
env = D1wCpgEnv(F0S_np, JINVS_np)

# MJX 上踩過的坑（task4 地形版）：actuator 的 biastype 若不是 affine，
# ctrl 會被當成力矩直接施加，機器人會直接塌掉。XML 已宣告 affine，
# 這裡確認 mjx.put_model 之後仍然保持。
assert env.sys.actuator_biastype[0] == mujoco.mjtBias.mjBIAS_AFFINE, \
    "actuator biastype 不是 affine，ctrl 會被當力矩施加 → 機器人會塌掉"

s = jax.jit(env.reset)(jax.random.PRNGKey(0))
print("obs shape:", s.obs.shape, "(應為 (69,))")
s = jax.jit(env.step)(s, jnp.zeros(12))
print("reward:", float(s.reward), "done:", float(s.done),
      "height:", float(s.metrics["height"]),
      "pitch:", float(s.metrics["pitch"]), "clr:", float(s.metrics["clr"]),
      "vz:", float(s.metrics["vz"]))
assert s.obs.shape == (69,), "obs 維度不對，回頭對 _obs 的欄位順序"
assert np.isfinite(float(s.reward)), "reward 出現 NaN/Inf"

# 把 env 真的丟進 brax 的 EvalWrapper 走一次。
# num_evals>1 時訓練會先跑一次 eval，metrics 少了 "reward" 鍵就會在
# EpisodeWrapper 內層的 lax.scan 炸 TypeError（見 Cell 6 reset 的註解）。
# 這幾秒鐘的檢查，換掉「編譯十分鐘後才死在 Cell 9」。
from brax.envs import training as _brax_training
_ck = _brax_training.EvalWrapper(
    _brax_training.EpisodeWrapper(env, episode_length=8, action_repeat=1))
_s0 = jax.jit(_ck.reset)(jax.random.PRNGKey(0))
_s1 = jax.jit(_ck.step)(_s0, jnp.zeros(12))
assert set(_s0.metrics) == set(_s1.metrics), (
    f"reset/step 的 metrics 鍵不一致：{sorted(_s0.metrics)} vs {sorted(_s1.metrics)}"
)
print("EvalWrapper OK, metrics keys:", sorted(_s1.metrics))
print("PASSED")

In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env = D1wCpgEnv(F0S_np, JINVS_np)
network_factory = functools.partial(
    ppo_networks.make_ppo_networks,
    policy_hidden_layer_sizes=(256, 256, 128),
    value_hidden_layer_sizes=(256, 256, 256))

# num_timesteps 120M → 60M（reward v2）。
# 依據：第一輪 120M 的 eval reward 在 25.6M=2378、44.7M=2418、57.5M=2461 之後就進入平台，
# 64M~121M 全程只在 2416~2461 之間震盪（±2%），約 50M 之後基本上是雜訊，
# 多跑的 60M 只是燒 GPU 時數。num_evals 維持 20（曲線點數不變，只是間隔縮一半）。
train_fn = functools.partial(
    ppo.train, num_timesteps=60_000_000, num_evals=20, episode_length=1000,
    num_envs=2048, batch_size=256, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.97, normalize_observations=True,
    network_factory=network_factory, randomization_fn=domain_randomize, seed=0)

_t0 = time.time(); rewards = []
def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); rewards.append((step, r))
    # pitch / clr / vz 除以 episode 長度 = 每控制步的平均值。
    # pitch：機身俯仰約略角度(度)，越小越好（v1 4–5°、v3 2.5°）。
    # clr：四腿平均離地淨空(mm)，越大越好（v1 ≈15、v3 ≈10、開迴路 ≈9）。
    # vz：機身垂直速度(m/s)，★v4 主目標，越小越好（v1 0.264、v3 0.414 ← 要壓回 0.25 以下）。
    # 這是 reward v2 的觀察指標：若它一路下降而 reward 不掉，就是前腳真的開始跨步了。
    L = float(metrics.get("eval/avg_episode_length", 1.0)) or 1.0
    pitch = float(metrics.get("eval/episode_pitch", 0.0)) / L
    clr = float(metrics.get("eval/episode_clr", 0.0)) / L
    vz = float(metrics.get("eval/episode_vz", 0.0)) / L
    print(f"step {step:>10,}  reward {r:8.2f}  pitch {pitch:5.2f}°  clr {clr:5.1f}mm  "
          f"vz {vz:5.3f}  ({time.time()-_t0:.0f}s)")

make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")

In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s, _ in rewards], [r for _, r in rewards], marker="o")
plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()

In [ ]:
from brax.io import model
model.save_params("cpg_rl_d1w_params.pkl", params)
try:
    from google.colab import files; files.download("cpg_rl_d1w_params.pkl")
except Exception as e:
    print("左側檔案面板右鍵下載 cpg_rl_d1w_params.pkl。", e)